# Setting up Controller and Event Analyst

Hello! This tutorial will show you how to set up `Ralph` to perform the tasks for you!

`Ralph` can:
* pre-process the light curves to make sure the fitting process runs smoothly;
* fit ongoing and fished gravitational microlensing events;
* create a color-magnitude diagram for all found best-fitting models.

`Ralph` is pretty flexible, and you can and may turn on and off certain modules.

## Overview

Ralph has two basic classes, a Controller and an Analyst.

The **Controller** manages the workflow for all events you'd like to analyze. It controls how many events at once are handled,
where the inputs and output files go, and how, where, at what level the logs are created. For each event you'd like to analyze,
the Controller will launch an **Event Analyst**.

The **Event Analyst** controls the analysis work flow for a single event. It handles which processes are launched based on the Event Analyst
configuration for each event. It knows where to store the outputs and the logs.
For each event you may or may not have different Event Analyst set up. If you have a set of events with different
light curve quality, you can set up their pre-processing or fitting procedures differently.
Or, if you have color-magnitude diagram information for only some of the events, or only some of the surveys, you can set up too!
Each event can be treated individually!

The **Event Analyst** manages three types of Analysts:
* the **Light Curve Analyst** -- it's handling the pre-processing of the light curves, to make sure no invalid entries prevent
running the fitting process smoothly;
* the **Fit Analyst** -- it's handling the fitting procedure, which is described in depth **here (add a document describing the workflow, at some point)**;
* the **CMD Analyst** -- it's handling the creation of the color-magnitude diagram for each event and catalog you have specified.


## Configuring the Controller

Okay! Let's start with something simple and set up our controller to run on a single event.
First we have to let `Ralph` know how do we set up the Controller.
You can pass a YAML or JSON file with the configuration, but you can also pass it as a dictionary.
We will create a dictionary, to make it easier in this tutorial.

In [1]:
import os

# Here we set up the path to the tutorial folder, to ensure that the example executes correctly
current_path = os.getcwd()
os.chdir("../../")
ralph_home_dir = os.getcwd()

In [2]:
config = {
    # Name of the python compiler you want to use for this run.
    # It can be just "python" to use a default compiler, but you
    # can also specify python3, or python3.10, python3.12, etc.
    "python_compiler": "python",
    # Group processing limit refers to how many events will be
    # analyzed at the same time. It is limited by how many cores
    # are available on your machine.
    "group_processing_limit": 2,
    # What file format (YAML or JSON) will you use for your configuration
    # files for the Event Analysts.
    "config_type": "yaml",
    # Where will the outputs be saved and where Event Analysts
    # should look for the configuration files for each event.
    # You can pass it as a string, or use the power of the os
    # package, when creating a dictionary.
    "events_path": os.path.join(ralph_home_dir, "examples", "controller"),
    # Because Ralph uses the subprocess package, it requires specific set up.
    # You have to specify what is the path of your Ralph's repository location
    # of the Analysts' source files.
    "software_dir": os.path.join(ralph_home_dir, "src", "ralph", "analyst"),
    # Ralph uses logging.Logger instance to handle logging statements for
    # Controller. If log_stream is set to "True", it can be streamed to
    # the standard output, and then captured with for example Kubernetes.
    # If set to false, the log will be saved to the location specified
    # in log_location.
    "log_stream": False,
    # Where you'd like to save the file with controller logs. The file
    # will appear in the log_location directory, under the name
    # "controller.log"
    "log_location": os.path.join(ralph_home_dir, "examples", "controller"),
    # How verbose the log statements should be? There are three levels
    # available: "error" (least verbose), "info" (medium), "debug" (very
    # verbose).
    "log_level": "info",
}

On top of the configuration information, `Ralph` also needs to know which events you'd like to have analyzed.
For the sake of simplicity, we will analyze only one event.

These event names will correspond to folder names inside the `events_path` location.

* If you pass the Event Analyst configuration as dictionaries created on the fly,
`Ralph` will create folders with event names inside the `events_path` directory.
* If you pass the Event Analyst configuration as files, they have to be located inside the
`events_path` folder, in a folder with the same name as your event name, under `config.yaml`
or `config.json` (whichever file format you specified in the Controller configuration under
`config_type`).

In [3]:
# We will analyze only one event from the OGLE survey
event_list = [
    "OGLE_2016_BLG_1395",
]

Now we can initialize the Controller.

In [4]:
from ralph.controller.controller import Controller

controller = Controller(event_list, config_dict=config)

In [5]:
# for key in controller.config:
#     print(key, ":", controller.config[key])

python_compiler : python
group_processing_limit : 2
config_type : yaml
events_path : /home/katarzyna/Documents/Microlensing_Fitting_Pipeline/new/ralph/examples/controller
software_dir : /home/katarzyna/Documents/Microlensing_Fitting_Pipeline/new/ralph/src/ralph/analyst
log_stream : False
log_location : /home/katarzyna/Documents/Microlensing_Fitting_Pipeline/new/ralph/examples/controller
log_level : info


Great! Now we have to set up our Event Analyst configuration and input files, before we can move forward.

## Configuring the Event Analyst -- the bare minimum

Let's say you only want to perform some basic fitting with `Ralph` for your event. 
We will show you what the minimal information is necessary to launch an event fitting.

This configuration file can, but doesn't have to include the following keywords:
* `lc_analyst` - it should hold a dictionary with the Light Curve Analyst set up.
If you leave this as an empty dictionary, it will launch the Light Curve Analyst with the default setup.
You can also drop this keyword completely, to not run the Light Curve Analyst at all, but it's not recommended
if you want to run the Fit Analyst.
* `fit_analyst` - it should hold a dictionary with the Fit Analyst set up. You can leave it empty, and it will
use the default setup for fitting. The default is `pyLIMA` as a fitting package, and
the [**Trust Region Reflective**](https://github.com/ebachelet/pyLIMA/blob/master/pyLIMA/fits/TRF_fit.py) fitting
routine for all types of models, except the initial one, where the [**Differential Evolution**](https://github.com/ebachelet/pyLIMA/blob/master/pyLIMA/fits/DE_fit.py) is used.
You can also drop this keyword completely, to not run the Fit Analyst at all,
* `cmd_analyst` - it should hold a dictionary with the CMD Analyst set up. Unlike the two previous Analysts, it doesn't have a default setup.
If you'd like to run the CMD Analyst for an event, you **have** to provide the configuration.

In this example, we will run the Light Curve Analyst with the default setup, and we will customize the Fit Analyst (in a moment).

In [5]:
event_analyst_config = {
    # Event name, should match an event name provided in "event_list".
    "event_name": "OGLE_2016_BLG_1395", # Mandatory
    # Event Right Ascension, in degrees
    "ra": 271.1194, # Mandatory
    # Event Declination, in degrees
    "dec": -29.8162, # Mandatory
    # A dictionary with the Light Curve Analyst setup.
    # We will leave it empty, to run with the default setup.
    "lc_analyst": {}, # Optional field
    # A dictionary with the Fit Analyst setup.
    # We will leave it empty for now, because it's the most complex one.
    "fit_analyst": {}, # Optional field
    # A list of dictionaries with light curves.
    "light_curves": [ # Mandatory
        {
            # Survey or telescope name, which was used to obtain the data.
            "survey": "OGLE", # Mandatory
            # Band (or filer) in which the data was obtained.
            "band": "I", # Mandatory
            # Path to a location where the light curve is stored.
            "path": os.path.join(ralph_home_dir, "examples", "example_light_curve_OB161395_I.dat") # Mandatory
        },
    ]
}

Now let's focus on the configuration of the Fit Analyst. It can get quite complex, so we will go step by step.

In [6]:
event_analyst_config["fit_analyst"] = {}